# The Complete GPT Model

We have built every piece. The tokenizer. The embeddings. RoPE
for position. Attention for context. And the transformer block
that packages everything into a repeatable unit.

Now we assemble it all into one model. A single class that takes
token IDs as input and predicts the next token as output. This
is the same architecture behind GPT-2 GPT-3 LLaMA and Mistral.

The model has four main parts. A token embedding that turns IDs
into vectors. A stack of transformer blocks that process those
vectors through attention and feed forward layers. A final
normalization layer. And an output head that projects the hidden
state back into vocabulary scores.

One clever trick we use is weight tying. The embedding table and
the output head share the same weights. This saves a third of
the parameters and gives the model more training signal because
each token embedding gets updated from both directions.

By the end of this notebook you will hold a complete language
model in your hands. Ready to train.

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

## Building blocks from previous chapters

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)


def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        attn_output = attn_weights @ v
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)
        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## GPT Configuration

A single dataclass holds every setting. Change one number and
the whole model adapts.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    d_model: int = 768
    num_heads: int = 12
    num_layers: int = 12
    max_seq_len: int = 1024
    dropout: float = 0.1
    embd_dropout: float = 0.1
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_steps: int = 2000
    max_steps: int = 100000
    batch_size: int = 8
    grad_accum_steps: int = 4
    betas: tuple = (0.9, 0.95)
    eps: float = 1e-8

## The GPT Model

Token embedding plus dropout. A stack of transformer blocks.
Final RMSNorm. Output projection into vocabulary scores.
The embedding weights are shared with the output head.
All weights start from a normal distribution with std 0.02.

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.embd_dropout = nn.Dropout(config.embd_dropout)

        self.layers = nn.ModuleList([
            TransformerBlock(config.d_model, config.num_heads, config.dropout)
            for _ in range(config.num_layers)
        ])

        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if hasattr(module, 'bias') and module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, input_ids, targets=None):
        batch_size, seq_len = input_ids.shape

        x = self.token_embedding(input_ids)
        x = self.embd_dropout(x)

        mask = create_causal_mask(seq_len, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            logits_flat = logits[:, :-1, :].contiguous().view(-1, self.config.vocab_size)
            targets_flat = targets[:, 1:].contiguous().view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

## Create the model and inspect it

In [ ]:
config = GPTConfig()
model = GPT(config)
print(f"Total parameters: {model.get_num_params():,}")
print()

print(f"Config:")
print(f"  d_model:   {config.d_model}")
print(f"  num_heads: {config.num_heads}")
print(f"  num_layers:{config.num_layers}")
print(f"  vocab_size:{config.vocab_size}")
print(f"  max_seq_len: {config.max_seq_len}")

## Run a forward pass

Feed the model some random token IDs and check that it produces
logits of the right shape. Also compute the loss.

In [ ]:
batch_size = 4
seq_len = 128

input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
target_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))

logits, loss = model(input_ids, target_ids)

print(f"Input shape:  {input_ids.shape}")
print(f"Logits shape: {logits.shape}")
print(f"Loss:         {loss.item():.4f}")
print()

print(f"Expected logits shape: [{batch_size}, {seq_len}, {config.vocab_size}]")
print(f"Actual logits shape:   [{logits.shape[0]}, {logits.shape[1]}, {logits.shape[2]}]")
print()

print(f"For each of the {seq_len} positions the model produces")
print(f"{config.vocab_size} scores. One score per possible next token.")

## Weight tying verification

The embedding and output head share the same tensor. Changing
one changes the other because they point to the same memory.

In [ ]:
emb_weight = model.token_embedding.weight
head_weight = model.lm_head.weight

print(f"Embedding weight id: {id(emb_weight)}")
print(f"LM head weight id:   {id(head_weight)}")
print(f"Same memory:         {emb_weight.data_ptr() == head_weight.data_ptr()}")
print()

saved_params = config.vocab_size * config.d_model
print(f"Parameters saved by weight tying: {saved_params:,}")